# Recodificación de la variable `cobertura` en las Slope Units

**Objetivo.** Dejar la variable de cobertura del suelo lista para entrar al modelo de susceptibilidad, con dos transformaciones:

1. **Renombrar** las etiquetas `Clase_0 … Clase_5` por nombres legibles (CLC).
2. **Reasignar** la clase `4 = Aguas continentales` a `1 = Tejido urbano discontinuo`, porque por su escasez (30 polígonos) y por no ser una superficie deslizable, no aporta señal al modelo. Tras esa fusión se renumera la clase `5 = Tierras desnudas y degradadas` a `4`, dejando un dominio limpio `{0,1,2,3,4}`.

Los `NaN` se preservan en ambos pasos (no se imputan).

**Insumo.** `slope_units_final.gpkg`, capa `slope_units_final` (61.242 SU).  
**Salida.** `slope_units_cobertura_recodificada.gpkg`, misma capa, columnas `cobertura` y `cobertura_cat` actualizadas.

## 1. Imports y configuración

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

INPUT_PATH  = Path('slope_units_final.gpkg')
INPUT_LAYER = 'slope_units_final'

OUTPUT_PATH  = Path('slope_units_cobertura_recodificada.gpkg')
OUTPUT_LAYER = 'slope_units_final'

COL_STR = 'cobertura'      # string: 'Clase_0', ..., 'Clase_5', NaN
COL_NUM = 'cobertura_cat'  # numérica: 0..5, NaN

## 2. Carga del GeoPackage e inspección inicial

Verificamos CRS, tamaño y distribuciones para tener un *baseline* contra el cual validar después.

In [2]:
gdf = gpd.read_file(INPUT_PATH, layer=INPUT_LAYER)

print(f'Filas        : {len(gdf):,}')
print(f'CRS          : {gdf.crs}')
print(f'Geometría    : {gdf.geom_type.unique()}')
print(f'Columnas     : {gdf.columns.tolist()}')

Filas        : 61,242
CRS          : EPSG:3116
Geometría    : ['MultiPolygon']
Columnas     : ['Id', 'gridcode', 'Shape_Leng', 'Shape_Area', 'slope_mean', 'curva_mean', 'acos_mean', 'asin_mean', 'aspect_mea', 'area', 'su_type', 'su_id', 'Codigo', 'Unidad_geo', 'Granulomet', 'Espesor_m', 'Gamma_kNm3', 'C_kPa', 'Phi', 'cobertura_cat', 'cobertura', 'n_mm', 'si_no', 'suelos', 'zona_lluvia', 'geometry']


In [3]:
def resumen_cobertura(gdf, col_str=COL_STR, col_num=COL_NUM):
    """Devuelve una tabla con conteos cruzados de las dos columnas de cobertura.

    Útil para validar consistencia entre la versión textual y la numérica
    antes y después de cada transformación.
    """
    tabla = (
        gdf[[col_num, col_str]]
        .value_counts(dropna=False)
        .rename('n')
        .reset_index()
        .sort_values(col_num, na_position='last')
        .reset_index(drop=True)
    )
    return tabla

print('--- Estado inicial: pares (cobertura_cat, cobertura) ---')
resumen_cobertura(gdf)

--- Estado inicial: pares (cobertura_cat, cobertura) ---


,cobertura_cat,cobertura,n
0,0.0,Clase_0,3744
1,1.0,Clase_1,11824
2,2.0,Clase_2,17770
3,3.0,Clase_3,27190
4,4.0,Clase_4,30
5,5.0,Clase_5,437
6,NaN,NaN,247


## 3. Diccionarios de mapeo

Centralizamos las equivalencias en diccionarios para que la lógica sea **explícita, auditable y reproducible**.

- `MAPEO_CLASE_A_NOMBRE`: traducción literal `Clase_X` → nombre CLC.
- `MAPEO_REASIGNACION_NUM`: regla de fusión y renumeración (`4 → 1`, `5 → 4`).
- `MAPEO_COD_FINAL_A_NOMBRE`: codificación final tras la reasignación.

In [4]:
MAPEO_CLASE_A_NOMBRE = {
    'Clase_0': 'Tejido urbano continuo',
    'Clase_1': 'Tejido urbano discontinuo',
    'Clase_2': 'Territorios agrícolas',
    'Clase_3': 'Bosques',
    'Clase_4': 'Aguas continentales',
    'Clase_5': 'Tierras desnudas y degradadas',
}

MAPEO_REASIGNACION_NUM = {
    0: 0,  # Tejido urbano continuo            -> sin cambio
    1: 1,  # Tejido urbano discontinuo         -> sin cambio
    2: 2,  # Territorios agrícolas             -> sin cambio
    3: 3,  # Bosques                           -> sin cambio
    4: 1,  # Aguas continentales               -> Tejido urbano discontinuo (fusión)
    5: 4,  # Tierras desnudas y degradadas     -> renumeración (5 -> 4)
}

MAPEO_COD_FINAL_A_NOMBRE = {
    0: 'Tejido urbano continuo',
    1: 'Tejido urbano discontinuo',
    2: 'Territorios agrícolas',
    3: 'Bosques',
    4: 'Tierras desnudas y degradadas',
}

## 4. Paso 1 — Renombrar `Clase_X` por nombres legibles

Cambia únicamente la columna `cobertura` (string). El método `Series.map` con un diccionario asigna `NaN` a cualquier valor no listado, pero como aquí cubrimos todas las clases existentes, el único `NaN` resultante son los `NaN` originales, que se preservan.

In [5]:
def renombrar_clases(
    gdf: gpd.GeoDataFrame,
    mapeo: dict,
    col_str: str = COL_STR,
) -> gpd.GeoDataFrame:
    """Renombra los valores `Clase_X` de la columna textual por nombres legibles.

    Parameters
    ----------
    gdf : GeoDataFrame de entrada (no se modifica in-place).
    mapeo : dict con la equivalencia 'Clase_X' -> nombre.
    col_str : nombre de la columna textual de cobertura.

    Returns
    -------
    GeoDataFrame con la columna `col_str` renombrada y los NaN preservados.
    """
    out = gdf.copy()

    # Validación: ¿hay valores no contemplados (ni NaN ni en el mapeo)?
    valores = out[col_str].dropna().unique()
    no_mapeados = [v for v in valores if v not in mapeo]
    if no_mapeados:
        raise ValueError(
            f'Valores no contemplados en {col_str}: {no_mapeados}. '
            f'Revisar el diccionario MAPEO_CLASE_A_NOMBRE.'
        )

    # map() preserva NaN de forma natural
    out[col_str] = out[col_str].map(mapeo).astype('object')
    return out


gdf_p1 = renombrar_clases(gdf, MAPEO_CLASE_A_NOMBRE)

print('--- Tras Paso 1: cobertura (string) renombrada ---')
resumen_cobertura(gdf_p1)

--- Tras Paso 1: cobertura (string) renombrada ---


,cobertura_cat,cobertura,n
0,0.0,Tejido urbano continuo,3744
1,1.0,Tejido urbano discontinuo,11824
2,2.0,Territorios agrícolas,17770
3,3.0,Bosques,27190
4,4.0,Aguas continentales,30
5,5.0,Tierras desnudas y degradadas,437
6,NaN,NaN,247


## 5. Paso 2 — Reasignar la columna numérica y sincronizar la textual

**Decisión justificada.** La clase `4 = Aguas continentales` se fusiona con `1 = Tejido urbano discontinuo` por dos razones: (i) sólo agrupa 30 SU (~0,05 % del dataset), insuficiente para estimar un coeficiente estable; (ii) los cuerpos de agua no son superficies deslizables, así que su valor mecánico es nulo en el modelo. Asignarla a la clase de fondo más cercana evita inflar la matriz de diseño con una categoría redundante.

Tras la fusión se renumera `5 → 4` para mantener un dominio compacto `{0,1,2,3,4}`.

**Sincronización.** La columna textual `cobertura` se regenera desde el código numérico nuevo, garantizando consistencia 1:1 entre `cobertura_cat` y `cobertura`.

In [6]:
def reasignar_codigos(
    gdf: gpd.GeoDataFrame,
    mapeo_num: dict,
    mapeo_cod_a_nombre: dict,
    col_num: str = COL_NUM,
    col_str: str = COL_STR,
) -> gpd.GeoDataFrame:
    """Aplica fusión/renumeración numérica y sincroniza la columna textual.

    Parameters
    ----------
    gdf : GeoDataFrame de entrada.
    mapeo_num : dict con la regla de reasignación de códigos numéricos.
    mapeo_cod_a_nombre : dict con la etiqueta legible para cada código nuevo.
    col_num : columna numérica de cobertura.
    col_str : columna textual de cobertura.

    Returns
    -------
    GeoDataFrame con `col_num` y `col_str` actualizadas y consistentes.
    Los NaN se preservan en ambas columnas.
    """
    out = gdf.copy()

    # 1) Validar que todos los códigos presentes (no-NaN) estén en el mapeo numérico
    presentes = (
        out[col_num].dropna().astype(int).unique().tolist()
    )
    no_mapeados = [c for c in presentes if c not in mapeo_num]
    if no_mapeados:
        raise ValueError(
            f'Códigos sin regla de reasignación en {col_num}: {no_mapeados}.'
        )

    # 2) Reasignar códigos numéricos. Mantener NaN como NaN -> dtype Float64 (nullable).
    nuevos = out[col_num].map(mapeo_num).astype('Float64')
    out[col_num] = nuevos

    # 3) Sincronizar la columna textual desde el código nuevo
    out[col_str] = nuevos.map(mapeo_cod_a_nombre).astype('object')

    return out


gdf_final = reasignar_codigos(
    gdf_p1,
    MAPEO_REASIGNACION_NUM,
    MAPEO_COD_FINAL_A_NOMBRE,
)

print('--- Estado final: pares (cobertura_cat, cobertura) ---')
resumen_cobertura(gdf_final)

--- Estado final: pares (cobertura_cat, cobertura) ---


,cobertura_cat,cobertura,n
0,0.0,Tejido urbano continuo,3744
1,1.0,Tejido urbano discontinuo,11854
2,2.0,Territorios agrícolas,17770
3,3.0,Bosques,27190
4,4.0,Tierras desnudas y degradadas,437
5,<NA>,NaN,247


## 6. Validaciones

Tres comprobaciones mínimas:

1. **Número de filas conservado** (no se pierden ni duplican SU).
2. **NaN preservados** (mismo conteo antes y después en ambas columnas).
3. **Conservación de masa por clase**: la suma de SU en las clases `1` y `4` finales debe igualar `1 + 4` originales (para `1`) y `5` original (para `4`), respectivamente. El resto debe coincidir.

In [7]:
def validar_transformacion(gdf_orig, gdf_nuevo, col_num=COL_NUM, col_str=COL_STR):
    """Comprueba conservación de filas, NaN y masa por clase tras la reasignación."""
    # 1. Filas
    assert len(gdf_orig) == len(gdf_nuevo), 'Cambió el número de filas.'

    # 2. NaN preservados
    nan_num_orig = gdf_orig[col_num].isna().sum()
    nan_num_new  = gdf_nuevo[col_num].isna().sum()
    nan_str_orig = gdf_orig[col_str].isna().sum()
    nan_str_new  = gdf_nuevo[col_str].isna().sum()
    assert nan_num_orig == nan_num_new, f'NaN en {col_num}: {nan_num_orig} -> {nan_num_new}'
    assert nan_str_orig == nan_str_new, f'NaN en {col_str}: {nan_str_orig} -> {nan_str_new}'

    # 3. Conservación de masa
    cnt_orig = gdf_orig[col_num].value_counts(dropna=False).sort_index()
    cnt_new  = gdf_nuevo[col_num].value_counts(dropna=False).sort_index()

    esperado = {
        0: cnt_orig.get(0, 0),
        1: cnt_orig.get(1, 0) + cnt_orig.get(4, 0),  # antiguo 4 -> 1
        2: cnt_orig.get(2, 0),
        3: cnt_orig.get(3, 0),
        4: cnt_orig.get(5, 0),                        # antiguo 5 -> 4
    }
    for k, v in esperado.items():
        obtenido = cnt_new.get(k, 0)
        assert obtenido == v, f'Clase {k}: esperado {v}, obtenido {obtenido}'

    print('OK — Filas, NaN y masa por clase conservados correctamente.')
    print(f'  · Total filas        : {len(gdf_nuevo):,}')
    print(f'  · NaN en {col_num:<14}: {nan_num_new}')
    print(f'  · NaN en {col_str:<14}: {nan_str_new}')


validar_transformacion(gdf, gdf_final)

OK — Filas, NaN y masa por clase conservados correctamente.
  · Total filas        : 61,242
  · NaN en cobertura_cat : 247
  · NaN en cobertura     : 247


In [8]:
# Tabla compacta de equivalencias para incluir en la tesis
tabla_equiv = pd.DataFrame(
    [(k, v, gdf_final[gdf_final[COL_NUM] == k].shape[0])
     for k, v in MAPEO_COD_FINAL_A_NOMBRE.items()],
    columns=['Código', 'Cobertura', 'N (SU)'],
)
tabla_equiv.loc[len(tabla_equiv)] = [
    np.nan, 'Sin dato (NaN)', int(gdf_final[COL_NUM].isna().sum())
]
tabla_equiv['%'] = (tabla_equiv['N (SU)'] / len(gdf_final) * 100).round(2)
tabla_equiv

,Código,Cobertura,N (SU),%
0,0.0,Tejido urbano continuo,3744,6.11
1,1.0,Tejido urbano discontinuo,11854,19.36
2,2.0,Territorios agrícolas,17770,29.02
3,3.0,Bosques,27190,44.40
4,4.0,Tierras desnudas y degradadas,437,0.71
5,NaN,Sin dato (NaN),247,0.40


## 7. Guardar el GeoPackage resultante

In [9]:
gdf_final.to_file(OUTPUT_PATH, layer=OUTPUT_LAYER, driver='GPKG')
print(f'Guardado en: {OUTPUT_PATH.resolve()}')
print(f'Capa       : {OUTPUT_LAYER}')
print(f'CRS        : {gdf_final.crs}')

TypeError: Cannot interpret 'Float64Dtype()' as a data type